# Percobaan 5 - Simple CatBoost Full Ensemble + Jalur A

Versi lanjutan dari `simple_catboost_baseline.ipynb`.

Target notebook ini:
- tetap memakai preprocessing simple yang sudah menghasilkan LB sekitar 3.14,
- train beberapa variasi CatBoost yang stabil,
- optional menambahkan LightGBM/XGBoost kalau library tersedia,
- memilih blend weight berdasarkan validation AW-MAE,
- Jalur A: outcome-aware post-processing + score calibration,
- generate submission ensemble Jalur A.

Catatan: kalau waktu mepet, jalankan dulu CatBoost ensemble + Jalur A. Model opsional otomatis di-skip kalau package tidak tersedia.

In [1]:
import warnings
warnings.filterwarnings('ignore')

from pathlib import Path
import time

import numpy as np
import pandas as pd

from catboost import CatBoostRegressor
from sklearn.preprocessing import OrdinalEncoder

pd.set_option('display.max_columns', 140)
pd.set_option('display.width', 180)

## 1. Path dan konfigurasi

`TASK_TYPE` default CPU supaya stabil. Kalau CatBoost GPU di kernel `py_gpu_ready` sudah aman, ganti ke `GPU`.

In [2]:
BASE_PATH = Path.home() / 'Downloads' / 'Gammafest'
DATA_PATH = BASE_PATH / 'dataset'
OUTPUT_DIR = BASE_PATH / 'experiments' / 'percobaan 5 - simple baseline'

TRAIN_PATH = DATA_PATH / 'train.csv'
TEST_PATH = DATA_PATH / 'test.csv'
SAMPLE_PATH = DATA_PATH / 'sample submission.csv'

SUBMISSION_PATH = OUTPUT_DIR / 'submission_full_ensemble_jalur_a.csv'
SUBMISSION_ROUNDCLIP_PATH = OUTPUT_DIR / 'submission_full_ensemble_roundclip.csv'
OOF_REPORT_PATH = OUTPUT_DIR / 'ensemble_validation_report.csv'
POSTPROCESS_REPORT_PATH = OUTPUT_DIR / 'jalur_a_postprocess_report.csv'

RANDOM_STATE = 42
VALID_FRAC = 0.20
TASK_TYPE = 'CPU'  # ganti ke 'GPU' kalau ingin coba CatBoost GPU
MAX_SCORE = 6
N_RANDOM_BLENDS = 2500

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print('Output:', OUTPUT_DIR)

Output: C:\Users\Gusti Jogish\Downloads\Gammafest\experiments\percobaan 5 - simple baseline


## 2. Load data

In [3]:
train_raw = pd.read_csv(TRAIN_PATH)
test_raw = pd.read_csv(TEST_PATH)
sample = pd.read_csv(SAMPLE_PATH)

print('train:', train_raw.shape)
print('test :', test_raw.shape)
print('sample:', sample.shape)
display(train_raw.head(2))
display(test_raw.head(2))

train: (78772, 47)
test : (42422, 20)
sample: (42422, 3)


,Id,match_id,date,gender,team,opponent,is_home,neutral,tournament,venue_country,team_goals,opp_goals,team_points_last5,opp_points_last5,points_last5_diff,team_gd_last5,opp_gd_last5,gd_last5_diff,h2h_points_last5,h2h_gd_last5,days_since_last_match_team,days_since_last_match_opp,team_points_last10,opp_points_last10,team_avg_goals_last5,team_avg_conceded_last5,opp_avg_goals_last5,opp_avg_conceded_last5,team_win_rate_last10,opp_win_rate_last10,elo_team,elo_opponent,rank_team,rank_opponent,rank_diff,rank_missing_team,rank_missing_opp,confederation_team,confederation_opp,population_team,population_opp,gdp_per_capita_team,gdp_per_capita_opp,altitude_venue,distance_travel_team,distance_travel_opp,temperature_venue
0,M000001_Scotland,M000001,1872-11-30,M,Scotland,England,1,0,Friendly,Scotland,0,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1500.0,1500.0,NaN,NaN,NaN,1,1,UEFA,UEFA,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,M000001_England,M000001,1872-11-30,M,England,Scotland,0,0,Friendly,Scotland,0,0,NaN,1.0,NaN,NaN,0.0,NaN,NaN,NaN,NaN,0.0,NaN,1.0,NaN,NaN,0.0,0.0,NaN,0.0,1500.0,1500.0,NaN,NaN,NaN,1,1,UEFA,UEFA,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


,Id,match_id,date,gender,team,opponent,is_home,neutral,tournament,venue_country,confederation_team,confederation_opp,population_team,population_opp,gdp_per_capita_team,gdp_per_capita_opp,altitude_venue,distance_travel_team,distance_travel_opp,temperature_venue
0,M034984_Seychelles,M034984,2011-08-06,M,Seychelles,Mauritius,1,0,Indian Ocean Island Games,Seychelles,CAF,CAF,92409.0,1283330.0,12189.095160,9197.026972,-9999.0,0.000000,1751.895724,25.790169
1,M034984_Mauritius,M034984,2011-08-06,M,Mauritius,Seychelles,0,0,Indian Ocean Island Games,Seychelles,CAF,CAF,1283330.0,92409.0,9197.026972,12189.095160,-9999.0,1751.895724,0.000000,25.790169


## 3. Fitur dan preprocessing simple

Ini sengaja sama dengan baseline: hanya fitur yang ada di train dan test, missing flag, median train, dan categorical string untuk CatBoost.

In [4]:
CAT_COLS = [
    'gender', 'team', 'opponent', 'tournament', 'venue_country',
    'confederation_team', 'confederation_opp',
]

NUM_COLS = [
    'is_home', 'neutral',
    'population_team', 'population_opp',
    'gdp_per_capita_team', 'gdp_per_capita_opp',
    'altitude_venue', 'distance_travel_team', 'distance_travel_opp', 'temperature_venue',
]

DATE_FEATURES = ['year', 'month', 'dayofweek']
TARGET_COLS = ['team_goals', 'opp_goals']


def add_date_features(df):
    df = df.copy()
    dt = pd.to_datetime(df['date'], errors='coerce')
    df['year'] = dt.dt.year
    df['month'] = dt.dt.month
    df['dayofweek'] = dt.dt.dayofweek
    return df


def prepare_features(train_df, test_df, cat_cols, num_cols):
    train = add_date_features(train_df)
    test = add_date_features(test_df)
    numeric_base = num_cols + DATE_FEATURES

    for df in [train, test]:
        for col in numeric_base:
            df[col] = pd.to_numeric(df[col], errors='coerce')
            df.loc[df[col] == -9999, col] = np.nan

    missing_flag_cols = []
    for col in numeric_base:
        flag_col = f'{col}_missing'
        train[flag_col] = train[col].isna().astype(int)
        test[flag_col] = test[col].isna().astype(int)
        missing_flag_cols.append(flag_col)

    medians = train[numeric_base].median(numeric_only=True)
    for col in numeric_base:
        fill_value = medians[col]
        if pd.isna(fill_value):
            fill_value = 0
        train[col] = train[col].fillna(fill_value)
        test[col] = test[col].fillna(fill_value)

    for col in cat_cols:
        train[col] = train[col].fillna('Unknown').astype(str)
        test[col] = test[col].fillna('Unknown').astype(str)

    feature_cols = cat_cols + numeric_base + missing_flag_cols
    return train, test, feature_cols, missing_flag_cols, medians

train_prep, test_prep, FEATURE_COLS, MISSING_FLAG_COLS, TRAIN_MEDIANS = prepare_features(train_raw, test_raw, CAT_COLS, NUM_COLS)
cat_feature_indices = [FEATURE_COLS.index(c) for c in CAT_COLS]

print('Total features:', len(FEATURE_COLS))
print('Missing after preprocessing:', train_prep[FEATURE_COLS].isna().sum().sum(), test_prep[FEATURE_COLS].isna().sum().sum())
print(FEATURE_COLS)

Total features: 33
Missing after preprocessing: 0 0
['gender', 'team', 'opponent', 'tournament', 'venue_country', 'confederation_team', 'confederation_opp', 'is_home', 'neutral', 'population_team', 'population_opp', 'gdp_per_capita_team', 'gdp_per_capita_opp', 'altitude_venue', 'distance_travel_team', 'distance_travel_opp', 'temperature_venue', 'year', 'month', 'dayofweek', 'is_home_missing', 'neutral_missing', 'population_team_missing', 'population_opp_missing', 'gdp_per_capita_team_missing', 'gdp_per_capita_opp_missing', 'altitude_venue_missing', 'distance_travel_team_missing', 'distance_travel_opp_missing', 'temperature_venue_missing', 'year_missing', 'month_missing', 'dayofweek_missing']


## 4. Temporal validation split

In [5]:
train_prep['date_dt'] = pd.to_datetime(train_prep['date'], errors='coerce')
train_sorted = train_prep.sort_values('date_dt').reset_index(drop=True)

split_idx = int(len(train_sorted) * (1 - VALID_FRAC))
tr_df = train_sorted.iloc[:split_idx].copy()
val_df = train_sorted.iloc[split_idx:].copy()

X_tr = tr_df[FEATURE_COLS]
X_val = val_df[FEATURE_COLS]
y_tr_team = tr_df['team_goals']
y_tr_opp = tr_df['opp_goals']
y_val_team = val_df['team_goals']
y_val_opp = val_df['opp_goals']

X_full = train_prep[FEATURE_COLS]
y_full_team = train_prep['team_goals']
y_full_opp = train_prep['opp_goals']
X_test = test_prep[FEATURE_COLS]

print('Train fold:', tr_df.shape, tr_df['date_dt'].min(), '->', tr_df['date_dt'].max())
print('Valid fold:', val_df.shape, val_df['date_dt'].min(), '->', val_df['date_dt'].max())

Train fold: (63017, 64) 1872-11-30 00:00:00 -> 2005-02-01 00:00:00
Valid fold: (15755, 64) 2005-02-01 00:00:00 -> 2011-08-04 00:00:00


## 5. AW-MAE dan basic post-processing

In [6]:
TOURNAMENT_WEIGHTS = {
    'FIFA World Cup': 2.00,
    'AFC Championship': 1.80,
    'AFC Asian Cup': 1.80,
    'UEFA Euro': 1.80,
    'Copa America': 1.80,
    'Copa Am?rica': 1.80,
    'Africa Cup of Nations': 1.80,
    'African Cup of Nations': 1.80,
    'Gold Cup': 1.75,
    'CONCACAF Gold Cup': 1.75,
    'FIFA World Cup qualification': 1.50,
    'UEFA Euro qualification': 1.40,
    'AFC Asian Cup qualification': 1.40,
    'Friendly': 0.96,
}
DEFAULT_TOURNAMENT_WEIGHT = 1.20


def outcome_label(team_goals, opp_goals):
    diff = np.asarray(team_goals) - np.asarray(opp_goals)
    return np.where(diff > 0, 1, np.where(diff < 0, -1, 0))


def postprocess_round_clip(team_pred, opp_pred, max_score=6):
    team = np.rint(team_pred).astype(int)
    opp = np.rint(opp_pred).astype(int)
    team = np.clip(team, 0, max_score)
    opp = np.clip(opp, 0, max_score)
    return team, opp


def compute_awmae(df_true, team_pred, opp_pred):
    true_team = df_true['team_goals'].to_numpy()
    true_opp = df_true['opp_goals'].to_numpy()
    pred_team = np.asarray(team_pred)
    pred_opp = np.asarray(opp_pred)

    mae = (np.abs(true_team - pred_team) + np.abs(true_opp - pred_opp)) / 2
    exact = ((true_team == pred_team) & (true_opp == pred_opp)).astype(int)
    outcome = (outcome_label(true_team, true_opp) == outcome_label(pred_team, pred_opp)).astype(int)
    gd = ((true_team - true_opp) == (pred_team - pred_opp)).astype(int)

    penalty = 0.30 * (1 - exact) + 0.25 * (1 - outcome) + 0.15 * (1 - gd)
    multiplier = np.where(outcome == 1, 1.0, 1.5)
    loss = ((mae + penalty) * multiplier) ** 1.5
    weights = df_true['tournament'].map(TOURNAMENT_WEIGHTS).fillna(DEFAULT_TOURNAMENT_WEIGHT).to_numpy()

    score = np.sum(loss * weights) / np.sum(weights)
    diag = {
        'AW-MAE': score,
        'MAE_raw_component': mae.mean(),
        'exact_acc': exact.mean(),
        'outcome_acc': outcome.mean(),
        'goal_diff_acc': gd.mean(),
    }
    return score, diag


def evaluate_predictions(name, df_true, team_raw, opp_raw, max_score=6):
    team_int, opp_int = postprocess_round_clip(team_raw, opp_raw, max_score=max_score)
    score, diag = compute_awmae(df_true, team_int, opp_int)
    out = {'model': name, **diag}
    return out, team_int, opp_int

## 6. CatBoost ensemble configs

Beberapa variasi kecil biasanya lebih aman daripada satu model besar. Semuanya masih memakai preprocessing simple.

In [7]:
CATBOOST_CONFIGS = [
    {
        'name': 'cat_mae_d7_seed42',
        'params': dict(loss_function='MAE', eval_metric='MAE', iterations=1400, learning_rate=0.045, depth=7, l2_leaf_reg=6, random_strength=1.0, bagging_temperature=0.4, random_seed=42),
    },
    {
        'name': 'cat_mae_d6_seed7',
        'params': dict(loss_function='MAE', eval_metric='MAE', iterations=1600, learning_rate=0.040, depth=6, l2_leaf_reg=8, random_strength=1.5, bagging_temperature=0.8, random_seed=7),
    },
    {
        'name': 'cat_mae_d8_seed99',
        'params': dict(loss_function='MAE', eval_metric='MAE', iterations=1200, learning_rate=0.035, depth=8, l2_leaf_reg=10, random_strength=0.8, bagging_temperature=0.2, random_seed=99),
    },
    {
        'name': 'cat_rmse_d7_seed123',
        'params': dict(loss_function='RMSE', eval_metric='MAE', iterations=1400, learning_rate=0.040, depth=7, l2_leaf_reg=7, random_strength=1.2, bagging_temperature=0.6, random_seed=123),
    },
]

BASE_CAT_PARAMS = dict(
    task_type=TASK_TYPE,
    verbose=150,
    allow_writing_files=False,
)

print('CatBoost configs:', [c['name'] for c in CATBOOST_CONFIGS])

CatBoost configs: ['cat_mae_d7_seed42', 'cat_mae_d6_seed7', 'cat_mae_d8_seed99', 'cat_rmse_d7_seed123']


## 7. Train validation ensemble

Cell ini bagian yang paling lama. Hasilnya dipakai untuk memilih blend weight terbaik.

In [9]:
val_pred_bank = {}
model_reports = []
trained_val_models = {}

start = time.time()
for cfg in CATBOOST_CONFIGS:
    name = cfg['name']
    params = {**BASE_CAT_PARAMS, **cfg['params']}
    print('' + '=' * 80)
    print('Training', name)

    model_t = CatBoostRegressor(**params)
    model_o = CatBoostRegressor(**params)

    model_t.fit(
        X_tr, y_tr_team,
        cat_features=cat_feature_indices,
        eval_set=(X_val, y_val_team),
        use_best_model=True,
        early_stopping_rounds=140,
    )
    model_o.fit(
        X_tr, y_tr_opp,
        cat_features=cat_feature_indices,
        eval_set=(X_val, y_val_opp),
        use_best_model=True,
        early_stopping_rounds=140,
    )

    pred_t = np.clip(model_t.predict(X_val), 0, None)
    pred_o = np.clip(model_o.predict(X_val), 0, None)
    val_pred_bank[name] = (pred_t, pred_o)
    trained_val_models[name] = (model_t, model_o)

    report, _, _ = evaluate_predictions(name, val_df, pred_t, pred_o, max_score=MAX_SCORE)
    report['best_iter_team'] = model_t.best_iteration_
    report['best_iter_opp'] = model_o.best_iteration_
    model_reports.append(report)
    print(report)

print('Done in minutes:', (time.time() - start) / 60)
reports_df = pd.DataFrame(model_reports).sort_values('AW-MAE')
display(reports_df)

Training cat_mae_d7_seed42
0:	learn: 1.1707540	test: 1.1294111	best: 1.1294111 (0)	total: 135ms	remaining: 3m 9s
150:	learn: 1.0573917	test: 1.0477100	best: 1.0476214 (149)	total: 7.27s	remaining: 1m
300:	learn: 1.0326976	test: 1.0378234	best: 1.0378234 (300)	total: 14.8s	remaining: 53.9s
450:	learn: 1.0148249	test: 1.0340560	best: 1.0339438 (421)	total: 22.5s	remaining: 47.4s
600:	learn: 1.0009336	test: 1.0324205	best: 1.0323849 (598)	total: 32.5s	remaining: 43.3s
750:	learn: 0.9898969	test: 1.0325955	best: 1.0320404 (655)	total: 39.8s	remaining: 34.4s
Stopped by overfitting detector  (140 iterations wait)

bestTest = 1.032040431
bestIteration = 655

Shrink model to first 656 iterations.
0:	learn: 1.1710106	test: 1.1298761	best: 1.1298761 (0)	total: 51.9ms	remaining: 1m 12s
150:	learn: 1.0572932	test: 1.0462221	best: 1.0462221 (150)	total: 7.17s	remaining: 59.3s
300:	learn: 1.0338839	test: 1.0388689	best: 1.0388689 (300)	total: 14.6s	remaining: 53.2s
450:	learn: 1.0172813	test: 1.0349

,model,AW-MAE,MAE_raw_component,exact_acc,outcome_acc,goal_diff_acc,best_iter_team,best_iter_opp
1,cat_mae_d6_seed7,3.149387,1.013393,0.106252,0.507839,0.239733,842,1195
3,cat_rmse_d7_seed123,3.155430,1.047096,0.091400,0.546557,0.231672,1217,1398
2,cat_mae_d8_seed99,3.159140,1.012853,0.103650,0.504792,0.233450,977,1125
0,cat_mae_d7_seed42,3.163438,1.011520,0.106125,0.502317,0.236496,655,850


## 8. Optional LightGBM/XGBoost models

Kalau package tidak tersedia, cell ini otomatis skip. Model ini memakai ordinal encoding untuk categorical.

In [10]:
def make_encoded_data():
    encoder = OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1)
    Xtr_enc = X_tr.copy()
    Xval_enc = X_val.copy()
    Xfull_enc = X_full.copy()
    Xtest_enc = X_test.copy()

    encoder.fit(Xtr_enc[CAT_COLS])
    for X in [Xtr_enc, Xval_enc, Xfull_enc, Xtest_enc]:
        X[CAT_COLS] = encoder.transform(X[CAT_COLS]).astype('float32')
    return Xtr_enc, Xval_enc, Xfull_enc, Xtest_enc

Xtr_enc, Xval_enc, Xfull_enc, Xtest_enc = make_encoded_data()

try:
    from lightgbm import LGBMRegressor
    HAS_LGBM = True
except Exception as e:
    HAS_LGBM = False
    print('LightGBM skipped:', repr(e))

if HAS_LGBM:
    lgbm_configs = [
        ('lgbm_mae_seed42', dict(objective='regression_l1', n_estimators=1400, learning_rate=0.035, num_leaves=63, subsample=0.85, colsample_bytree=0.85, reg_lambda=6.0, random_state=42)),
        ('lgbm_mae_seed99', dict(objective='regression_l1', n_estimators=1200, learning_rate=0.040, num_leaves=47, subsample=0.90, colsample_bytree=0.80, reg_lambda=9.0, random_state=99)),
    ]
    for name, params in lgbm_configs:
        print('Training', name)
        mt = LGBMRegressor(**params, verbosity=-1)
        mo = LGBMRegressor(**params, verbosity=-1)
        mt.fit(Xtr_enc, y_tr_team)
        mo.fit(Xtr_enc, y_tr_opp)
        pred_t = np.clip(mt.predict(Xval_enc), 0, None)
        pred_o = np.clip(mo.predict(Xval_enc), 0, None)
        val_pred_bank[name] = (pred_t, pred_o)
        trained_val_models[name] = (mt, mo)
        report, _, _ = evaluate_predictions(name, val_df, pred_t, pred_o, max_score=MAX_SCORE)
        model_reports.append(report)
        print(report)

try:
    from xgboost import XGBRegressor
    HAS_XGB = True
except Exception as e:
    HAS_XGB = False
    print('XGBoost skipped:', repr(e))

if HAS_XGB:
    xgb_configs = [
        ('xgb_mae_seed42', dict(n_estimators=1200, learning_rate=0.035, max_depth=6, min_child_weight=8, subsample=0.85, colsample_bytree=0.85, reg_lambda=8.0, random_state=42)),
        ('xgb_mae_seed123', dict(n_estimators=1000, learning_rate=0.040, max_depth=5, min_child_weight=10, subsample=0.90, colsample_bytree=0.80, reg_lambda=10.0, random_state=123)),
    ]
    for name, params in xgb_configs:
        print('Training', name)
        base = dict(objective='reg:absoluteerror', tree_method='hist', n_jobs=-1)
        mt = XGBRegressor(**base, **params)
        mo = XGBRegressor(**base, **params)
        mt.fit(Xtr_enc, y_tr_team, verbose=False)
        mo.fit(Xtr_enc, y_tr_opp, verbose=False)
        pred_t = np.clip(mt.predict(Xval_enc), 0, None)
        pred_o = np.clip(mo.predict(Xval_enc), 0, None)
        val_pred_bank[name] = (pred_t, pred_o)
        trained_val_models[name] = (mt, mo)
        report, _, _ = evaluate_predictions(name, val_df, pred_t, pred_o, max_score=MAX_SCORE)
        model_reports.append(report)
        print(report)

reports_df = pd.DataFrame(model_reports).sort_values('AW-MAE')
reports_df.to_csv(OOF_REPORT_PATH, index=False)
display(reports_df)
print('Saved report:', OOF_REPORT_PATH)

Training lgbm_mae_seed42
{'model': 'lgbm_mae_seed42', 'AW-MAE': np.float64(3.307611480335848), 'MAE_raw_component': np.float64(1.0377340526816883), 'exact_acc': np.float64(0.10384005077753095), 'outcome_acc': np.float64(0.48721040939384325), 'goal_diff_acc': np.float64(0.2365598222786417)}
Training lgbm_mae_seed99
{'model': 'lgbm_mae_seed99', 'AW-MAE': np.float64(3.309607151506807), 'MAE_raw_component': np.float64(1.0332910187242146), 'exact_acc': np.float64(0.10580768010155506), 'outcome_acc': np.float64(0.47940336401142497), 'goal_diff_acc': np.float64(0.23814662012059665)}
Training xgb_mae_seed42
{'model': 'xgb_mae_seed42', 'AW-MAE': np.float64(3.353179672280502), 'MAE_raw_component': np.float64(1.0403681370993336), 'exact_acc': np.float64(0.10403046651856554), 'outcome_acc': np.float64(0.4755950491907331), 'goal_diff_acc': np.float64(0.23541732783243416)}
Training xgb_mae_seed123
{'model': 'xgb_mae_seed123', 'AW-MAE': np.float64(3.401745096445464), 'MAE_raw_component': np.float64(1

,model,AW-MAE,MAE_raw_component,exact_acc,outcome_acc,goal_diff_acc,best_iter_team,best_iter_opp
1,cat_mae_d6_seed7,3.149387,1.013393,0.106252,0.507839,0.239733,842.0,1195.0
3,cat_rmse_d7_seed123,3.155430,1.047096,0.091400,0.546557,0.231672,1217.0,1398.0
2,cat_mae_d8_seed99,3.159140,1.012853,0.103650,0.504792,0.233450,977.0,1125.0
0,cat_mae_d7_seed42,3.163438,1.011520,0.106125,0.502317,0.236496,655.0,850.0
4,lgbm_mae_seed42,3.307611,1.037734,0.103840,0.487210,0.236560,NaN,NaN
5,lgbm_mae_seed99,3.309607,1.033291,0.105808,0.479403,0.238147,NaN,NaN
6,xgb_mae_seed42,3.353180,1.040368,0.104030,0.475595,0.235417,NaN,NaN
7,xgb_mae_seed123,3.401745,1.043637,0.104221,0.461949,0.234275,NaN,NaN


Saved report: C:\Users\Gusti Jogish\Downloads\Gammafest\experiments\percobaan 5 - simple baseline\ensemble_validation_report.csv


## 9. Blend weight search

Kita cari kombinasi weight berdasarkan validation AW-MAE. Mulai dari average, single best, lalu random Dirichlet weights.

In [11]:
model_names = list(val_pred_bank.keys())
team_matrix = np.vstack([val_pred_bank[name][0] for name in model_names])
opp_matrix = np.vstack([val_pred_bank[name][1] for name in model_names])

print('Models in bank:', model_names)
print('Prediction matrix:', team_matrix.shape, opp_matrix.shape)


def score_blend(weights, max_score=MAX_SCORE):
    weights = np.asarray(weights, dtype=float)
    weights = weights / weights.sum()
    pred_t = np.average(team_matrix, axis=0, weights=weights)
    pred_o = np.average(opp_matrix, axis=0, weights=weights)
    pred_t_int, pred_o_int = postprocess_round_clip(pred_t, pred_o, max_score=max_score)
    score, diag = compute_awmae(val_df, pred_t_int, pred_o_int)
    return score, diag, pred_t_int, pred_o_int

blend_rows = []
best = {'score': np.inf, 'weights': None, 'name': None, 'diag': None}

for i, name in enumerate(model_names):
    w = np.zeros(len(model_names))
    w[i] = 1.0
    score, diag, _, _ = score_blend(w)
    blend_rows.append({'blend': f'single_{name}', 'AW-MAE': score, **{f'w_{n}': w[j] for j, n in enumerate(model_names)}})
    if score < best['score']:
        best = {'score': score, 'weights': w.copy(), 'name': f'single_{name}', 'diag': diag}

w = np.ones(len(model_names)) / len(model_names)
score, diag, _, _ = score_blend(w)
blend_rows.append({'blend': 'equal_average', 'AW-MAE': score, **{f'w_{n}': w[j] for j, n in enumerate(model_names)}})
if score < best['score']:
    best = {'score': score, 'weights': w.copy(), 'name': 'equal_average', 'diag': diag}

single_scores = []
for i, name in enumerate(model_names):
    w_single = np.zeros(len(model_names)); w_single[i] = 1
    s, _, _, _ = score_blend(w_single)
    single_scores.append(s)
w = 1 / np.maximum(np.array(single_scores), 1e-9)
w = w / w.sum()
score, diag, _, _ = score_blend(w)
blend_rows.append({'blend': 'inverse_awmae', 'AW-MAE': score, **{f'w_{n}': w[j] for j, n in enumerate(model_names)}})
if score < best['score']:
    best = {'score': score, 'weights': w.copy(), 'name': 'inverse_awmae', 'diag': diag}

rng = np.random.default_rng(RANDOM_STATE)
alpha_options = [0.25, 0.5, 1.0, 2.0]
for k in range(N_RANDOM_BLENDS):
    alpha = alpha_options[k % len(alpha_options)]
    w = rng.dirichlet(np.ones(len(model_names)) * alpha)
    score, diag, _, _ = score_blend(w)
    if k < 25 or score < best['score']:
        blend_rows.append({'blend': f'random_{k}_a{alpha}', 'AW-MAE': score, **{f'w_{n}': w[j] for j, n in enumerate(model_names)}})
    if score < best['score']:
        best = {'score': score, 'weights': w.copy(), 'name': f'random_{k}_a{alpha}', 'diag': diag}

blend_df = pd.DataFrame(blend_rows).sort_values('AW-MAE').reset_index(drop=True)
display(blend_df.head(20))

print('Best blend:', best['name'])
print('Best AW-MAE:', best['score'])
print('Diagnostics:', best['diag'])
print('Weights:')
for name, weight in sorted(zip(model_names, best['weights']), key=lambda x: -x[1]):
    print(f'  {name:<24} {weight:.5f}')

Models in bank: ['cat_mae_d7_seed42', 'cat_mae_d6_seed7', 'cat_mae_d8_seed99', 'cat_rmse_d7_seed123', 'lgbm_mae_seed42', 'lgbm_mae_seed99', 'xgb_mae_seed42', 'xgb_mae_seed123']
Prediction matrix: (8, 15755) (8, 15755)


,blend,AW-MAE,w_cat_mae_d7_seed42,w_cat_mae_d6_seed7,w_cat_mae_d8_seed99,w_cat_rmse_d7_seed123,w_lgbm_mae_seed42,w_lgbm_mae_seed99,w_xgb_mae_seed42,w_xgb_mae_seed123
0,random_1033_a0.5,3.083898,0.149651,0.275629,0.006308,0.335956,0.057303,0.168134,0.000084,0.006935
1,random_811_a2.0,3.084262,0.119076,0.156620,0.105606,0.380067,0.035344,0.070265,0.053779,0.079243
2,random_79_a2.0,3.085201,0.243700,0.078265,0.136244,0.329279,0.031901,0.093177,0.028533,0.058900
3,random_43_a2.0,3.088491,0.061234,0.235962,0.005119,0.435522,0.011933,0.075865,0.034552,0.139814
4,random_30_a1.0,3.088913,0.019240,0.348932,0.043544,0.366602,0.033450,0.056700,0.123587,0.007945
5,random_1_a0.5,3.096100,0.041741,0.143013,0.099918,0.493764,0.110321,0.003194,0.093807,0.014242
6,random_0_a0.25,3.102681,0.231270,0.406316,0.000050,0.215250,0.000173,0.012107,0.110073,0.024759
7,random_12_a0.25,3.103196,0.506175,0.000001,0.112283,0.307730,0.003562,0.068524,0.000028,0.001696
8,random_8_a0.25,3.104701,0.001971,0.668557,0.070346,0.234895,0.000103,0.000006,0.023892,0.000231
9,random_17_a0.5,3.106078,0.032388,0.370874,0.005844,0.552852,0.010914,0.014448,0.012442,0.000236


Best blend: random_1033_a0.5
Best AW-MAE: 3.083897760873677
Diagnostics: {'AW-MAE': np.float64(3.083897760873677), 'MAE_raw_component': np.float64(1.009965090447477), 'exact_acc': np.float64(0.10199936528086322), 'outcome_acc': np.float64(0.5239606474135196), 'goal_diff_acc': np.float64(0.23827356394795304)}
Weights:
  cat_rmse_d7_seed123      0.33596
  cat_mae_d6_seed7         0.27563
  lgbm_mae_seed99          0.16813
  cat_mae_d7_seed42        0.14965
  lgbm_mae_seed42          0.05730
  xgb_mae_seed123          0.00694
  cat_mae_d8_seed99        0.00631
  xgb_mae_seed42           0.00008


## 10. Cek max score clip

Kadang `MAX_SCORE=5`, `6`, atau `7` beda tipis. Kita pilih dari validation.

In [12]:
clip_rows = []
for max_score in range(4, 9):
    score, diag, _, _ = score_blend(best['weights'], max_score=max_score)
    clip_rows.append({'max_score': max_score, **diag})
clip_df = pd.DataFrame(clip_rows).sort_values('AW-MAE')
display(clip_df)
BEST_MAX_SCORE = int(clip_df.iloc[0]['max_score'])
print('BEST_MAX_SCORE:', BEST_MAX_SCORE)

,max_score,AW-MAE,MAE_raw_component,exact_acc,outcome_acc,goal_diff_acc
2,6,3.083898,1.009965,0.101999,0.523961,0.238274
3,7,3.084146,1.010187,0.101872,0.523961,0.238147
4,8,3.084263,1.010251,0.101872,0.523961,0.238147
1,5,3.086551,1.011044,0.101365,0.523961,0.238020
0,4,3.093349,1.013837,0.101872,0.523961,0.238527


BEST_MAX_SCORE: 6


## 10A. Jalur A: outcome-aware post-processing

Bagian ini masih memakai raw prediction dari full ensemble, tapi cara mengubahnya ke skor integer tidak lagi hanya `round + clip`.

Untuk setiap row, kita coba kandidat skor integer seperti `0-0` sampai `6-6`, lalu pilih kandidat dengan objective terbaik:
- dekat dengan prediksi raw,
- outcome tidak terlalu melawan sinyal raw,
- goal difference tidak terlalu jauh,
- distribusi skor masih masuk akal berdasarkan train fold.

Parameter objective dicari dari validation AW-MAE, jadi ini sekaligus score calibration ringan.

In [13]:
def build_score_pair_prior(df, max_score=6, smoothing=1.0):
    counts = np.full((max_score + 1, max_score + 1), smoothing, dtype=float)
    team = np.clip(df['team_goals'].round().astype(int).to_numpy(), 0, max_score)
    opp = np.clip(df['opp_goals'].round().astype(int).to_numpy(), 0, max_score)
    for tg, og in zip(team, opp):
        counts[tg, og] += 1.0
    return counts / counts.sum()


def soft_outcome_from_raw(team_raw, opp_raw, draw_margin=0.20):
    diff = np.asarray(team_raw) - np.asarray(opp_raw)
    return np.where(diff > draw_margin, 1, np.where(diff < -draw_margin, -1, 0))


def outcome_aware_postprocess(team_raw, opp_raw, params, prior_matrix=None):
    max_score = int(params.get('max_score', 6))
    draw_margin = float(params.get('draw_margin', 0.20))
    outcome_weight = float(params.get('outcome_weight', 0.10))
    gd_weight = float(params.get('gd_weight', 0.05))
    prior_weight = float(params.get('prior_weight', 0.02))

    team_raw = np.asarray(team_raw, dtype=float)
    opp_raw = np.asarray(opp_raw, dtype=float)

    candidates = np.array([(tg, og) for tg in range(max_score + 1) for og in range(max_score + 1)], dtype=int)
    cand_team = candidates[:, 0]
    cand_opp = candidates[:, 1]
    cand_diff = cand_team - cand_opp
    cand_outcome = outcome_label(cand_team, cand_opp)

    base_cost = (np.abs(team_raw[:, None] - cand_team[None, :]) + np.abs(opp_raw[:, None] - cand_opp[None, :])) / 2
    gd_cost = np.abs((team_raw - opp_raw)[:, None] - cand_diff[None, :])

    raw_outcome = soft_outcome_from_raw(team_raw, opp_raw, draw_margin=draw_margin)
    outcome_cost = (raw_outcome[:, None] != cand_outcome[None, :]).astype(float)

    total_cost = base_cost + gd_weight * gd_cost + outcome_weight * outcome_cost

    if prior_matrix is not None and prior_weight > 0:
        prior = np.clip(prior_matrix[cand_team, cand_opp], 1e-12, None)
        prior_cost = -np.log(prior)
        total_cost = total_cost + prior_weight * prior_cost[None, :]

    best_idx = np.argmin(total_cost, axis=1)
    return cand_team[best_idx].astype(int), cand_opp[best_idx].astype(int)


VAL_BLEND_WEIGHTS = best['weights'] / best['weights'].sum()
val_blend_team_raw = np.average(team_matrix, axis=0, weights=VAL_BLEND_WEIGHTS)
val_blend_opp_raw = np.average(opp_matrix, axis=0, weights=VAL_BLEND_WEIGHTS)

round_team_int, round_opp_int = postprocess_round_clip(val_blend_team_raw, val_blend_opp_raw, max_score=BEST_MAX_SCORE)
round_score, round_diag = compute_awmae(val_df, round_team_int, round_opp_int)
print('Round+clip baseline AW-MAE:', round_score)
print(round_diag)

max_score_candidates = sorted(set([
    int(BEST_MAX_SCORE),
    max(4, int(BEST_MAX_SCORE) - 1),
    min(8, int(BEST_MAX_SCORE) + 1),
]))

draw_margins = [0.00, 0.10, 0.20, 0.30, 0.40]
outcome_weights = [0.00, 0.05, 0.10, 0.20, 0.35]
gd_weights = [0.00, 0.03, 0.06, 0.10]
prior_weights = [0.00, 0.01, 0.03, 0.06]

pp_rows = []
best_pp = {
    'score': round_score,
    'params': {
        'max_score': int(BEST_MAX_SCORE),
        'draw_margin': None,
        'outcome_weight': 0.0,
        'gd_weight': 0.0,
        'prior_weight': 0.0,
        'mode': 'round_clip',
    },
    'diag': round_diag,
}

start = time.time()
for max_score in max_score_candidates:
    prior_matrix = build_score_pair_prior(tr_df, max_score=max_score, smoothing=1.0)
    for draw_margin in draw_margins:
        for outcome_weight in outcome_weights:
            for gd_weight in gd_weights:
                for prior_weight in prior_weights:
                    params = {
                        'max_score': max_score,
                        'draw_margin': draw_margin,
                        'outcome_weight': outcome_weight,
                        'gd_weight': gd_weight,
                        'prior_weight': prior_weight,
                        'mode': 'outcome_aware',
                    }
                    pred_team_int, pred_opp_int = outcome_aware_postprocess(
                        val_blend_team_raw,
                        val_blend_opp_raw,
                        params,
                        prior_matrix=prior_matrix,
                    )
                    score, diag = compute_awmae(val_df, pred_team_int, pred_opp_int)
                    pp_rows.append({**params, **diag})
                    if score < best_pp['score']:
                        best_pp = {'score': score, 'params': params.copy(), 'diag': diag.copy()}

pp_df = pd.DataFrame(pp_rows).sort_values('AW-MAE').reset_index(drop=True)
pp_df.to_csv(POSTPROCESS_REPORT_PATH, index=False)

BEST_PP_PARAMS = best_pp['params']
BEST_PP_SCORE = best_pp['score']
BEST_MAX_SCORE = int(BEST_PP_PARAMS['max_score'])

print('Search done in minutes:', (time.time() - start) / 60)
print('Best postprocess score:', BEST_PP_SCORE)
print('Best postprocess params:', BEST_PP_PARAMS)
print('Best diagnostics:', best_pp['diag'])
print('Saved postprocess report:', POSTPROCESS_REPORT_PATH)
display(pp_df.head(25))

Round+clip baseline AW-MAE: 3.083897760873677
{'AW-MAE': np.float64(3.083897760873677), 'MAE_raw_component': np.float64(1.009965090447477), 'exact_acc': np.float64(0.10199936528086322), 'outcome_acc': np.float64(0.5239606474135196), 'goal_diff_acc': np.float64(0.23827356394795304)}
Search done in minutes: 0.7650701403617859
Best postprocess score: 3.002770884133499
Best postprocess params: {'max_score': 6, 'draw_margin': 0.0, 'outcome_weight': 0.35, 'gd_weight': 0.03, 'prior_weight': 0.06, 'mode': 'outcome_aware'}
Best diagnostics: {'AW-MAE': np.float64(3.002770884133499), 'MAE_raw_component': np.float64(1.027324658838464), 'exact_acc': np.float64(0.09704855601396382), 'outcome_acc': np.float64(0.580704538241828), 'goal_diff_acc': np.float64(0.22818152967311964)}
Saved postprocess report: C:\Users\Gusti Jogish\Downloads\Gammafest\experiments\percobaan 5 - simple baseline\jalur_a_postprocess_report.csv


,max_score,draw_margin,outcome_weight,gd_weight,prior_weight,mode,AW-MAE,MAE_raw_component,exact_acc,outcome_acc,goal_diff_acc
0,6,0.0,0.35,0.03,0.06,outcome_aware,3.002771,1.027325,0.097049,0.580705,0.228182
1,7,0.0,0.35,0.03,0.06,outcome_aware,3.003076,1.027547,0.096858,0.580705,0.227991
2,5,0.0,0.35,0.03,0.06,outcome_aware,3.004816,1.028150,0.096414,0.580705,0.227801
3,6,0.0,0.35,0.00,0.06,outcome_aware,3.004891,1.030117,0.095589,0.581593,0.225960
4,6,0.0,0.35,0.06,0.03,outcome_aware,3.004911,1.027198,0.097175,0.580260,0.228308
5,7,0.0,0.35,0.00,0.06,outcome_aware,3.005095,1.030340,0.095335,0.581593,0.225706
6,6,0.0,0.35,0.03,0.03,outcome_aware,3.005187,1.028785,0.096033,0.581085,0.225643
7,7,0.0,0.35,0.06,0.03,outcome_aware,3.005244,1.027420,0.096985,0.580260,0.228118
8,7,0.0,0.35,0.03,0.03,outcome_aware,3.005341,1.028975,0.095843,0.581085,0.225452
9,6,0.0,0.35,0.10,0.06,outcome_aware,3.006779,1.023961,0.098064,0.574294,0.230149


## 11. Train final ensemble on all train

Model final dilatih ulang dengan semua train. Untuk CatBoost, iterations memakai best iteration dari validation + buffer kecil.

In [17]:
final_pred_bank = {}
final_models = {}

for cfg in CATBOOST_CONFIGS:
    name = cfg['name']
    if name not in model_names:
        continue

    val_t, val_o = trained_val_models[name]
    best_iter = int(max(getattr(val_t, 'best_iteration_', 800), getattr(val_o, 'best_iteration_', 800)) + 120)
    params = {**BASE_CAT_PARAMS, **cfg['params']}
    params['iterations'] = max(best_iter, 300)
    params.pop('eval_metric', None)

    print('Final training', name, 'iterations:', params['iterations'])
    mt = CatBoostRegressor(**params)
    mo = CatBoostRegressor(**params)
    mt.fit(X_full, y_full_team, cat_features=cat_feature_indices, verbose=150)
    mo.fit(X_full, y_full_opp, cat_features=cat_feature_indices, verbose=150)

    pred_t = np.clip(mt.predict(X_test), 0, None)
    pred_o = np.clip(mo.predict(X_test), 0, None)
    final_pred_bank[name] = (pred_t, pred_o)
    final_models[name] = (mt, mo)

if 'HAS_LGBM' in globals() and HAS_LGBM:
    for name, params in [
        ('lgbm_mae_seed42', dict(objective='regression_l1', n_estimators=1400, learning_rate=0.035, num_leaves=63, subsample=0.85, colsample_bytree=0.85, reg_lambda=6.0, random_state=42)),
        ('lgbm_mae_seed99', dict(objective='regression_l1', n_estimators=1200, learning_rate=0.040, num_leaves=47, subsample=0.90, colsample_bytree=0.80, reg_lambda=9.0, random_state=99)),
    ]:
        if name not in model_names:
            continue
        print('Final training', name)
        mt = LGBMRegressor(**params, verbosity=-1)
        mo = LGBMRegressor(**params, verbosity=-1)
        mt.fit(Xfull_enc, y_full_team)
        mo.fit(Xfull_enc, y_full_opp)
        final_pred_bank[name] = (np.clip(mt.predict(Xtest_enc), 0, None), np.clip(mo.predict(Xtest_enc), 0, None))
        final_models[name] = (mt, mo)

if 'HAS_XGB' in globals() and HAS_XGB:
    for name, params in [
        ('xgb_mae_seed42', dict(n_estimators=1200, learning_rate=0.035, max_depth=6, min_child_weight=8, subsample=0.85, colsample_bytree=0.85, reg_lambda=8.0, random_state=42)),
        ('xgb_mae_seed123', dict(n_estimators=1000, learning_rate=0.040, max_depth=5, min_child_weight=10, subsample=0.90, colsample_bytree=0.80, reg_lambda=10.0, random_state=123)),
        ]:
        if name not in model_names:
            continue
        print('Final training', name)
        base = dict(objective='reg:absoluteerror', tree_method='hist', n_jobs=-1)
        mt = XGBRegressor(**base, **params)
        mo = XGBRegressor(**base, **params)
        mt.fit(Xfull_enc, y_full_team, verbose=False)
        mo.fit(Xfull_enc, y_full_opp, verbose=False)
        final_pred_bank[name] = (np.clip(mt.predict(Xtest_enc), 0, None), np.clip(mo.predict(Xtest_enc), 0, None))
        final_models[name] = (mt, mo)

print('Final prediction bank:', list(final_pred_bank.keys()))

Final training cat_mae_d7_seed42 iterations: 970
0:	learn: 1.1628650	total: 56.5ms	remaining: 54.7s
150:	learn: 1.0498076	total: 8.32s	remaining: 45.1s
300:	learn: 1.0299408	total: 16.5s	remaining: 36.7s
450:	learn: 1.0117139	total: 27.5s	remaining: 31.7s
600:	learn: 0.9985444	total: 41.4s	remaining: 25.4s
750:	learn: 0.9891659	total: 55.5s	remaining: 16.2s
900:	learn: 0.9811088	total: 1m 8s	remaining: 5.27s
969:	learn: 0.9777475	total: 1m 15s	remaining: 0us
0:	learn: 1.1629667	total: 104ms	remaining: 1m 40s
150:	learn: 1.0490162	total: 14s	remaining: 1m 15s
300:	learn: 1.0271513	total: 28.3s	remaining: 1m 2s
450:	learn: 1.0106686	total: 42.5s	remaining: 48.9s
600:	learn: 0.9992190	total: 56.3s	remaining: 34.6s
750:	learn: 0.9892657	total: 1m 9s	remaining: 20.4s
900:	learn: 0.9811750	total: 1m 21s	remaining: 6.21s
969:	learn: 0.9773812	total: 1m 24s	remaining: 0us
Final training cat_mae_d6_seed7 iterations: 1315
0:	learn: 1.1635756	total: 51.7ms	remaining: 1m 7s
150:	learn: 1.0669014	t

## 12. Generate full ensemble Jalur A submission

In [20]:
missing_final = [name for name in model_names if name not in final_pred_bank]
if missing_final:
    raise ValueError(f'Missing final predictions for: {missing_final}')

test_team_matrix = np.vstack([final_pred_bank[name][0] for name in model_names])
test_opp_matrix = np.vstack([final_pred_bank[name][1] for name in model_names])
weights = best['weights'] / best['weights'].sum()

test_pred_team_raw = np.average(test_team_matrix, axis=0, weights=weights)
test_pred_opp_raw = np.average(test_opp_matrix, axis=0, weights=weights)

round_team_int, round_opp_int = postprocess_round_clip(test_pred_team_raw, test_pred_opp_raw, max_score=BEST_MAX_SCORE)
submission_round = sample[['Id']].copy()
submission_round['team_goals'] = round_team_int
submission_round['opp_goals'] = round_opp_int
submission_round.to_csv(SUBMISSION_ROUNDCLIP_PATH, index=False)

if 'BEST_PP_PARAMS' in globals() and BEST_PP_PARAMS.get('mode') == 'outcome_aware':
    full_prior_matrix = build_score_pair_prior(train_prep, max_score=int(BEST_PP_PARAMS['max_score']), smoothing=1.0)
    final_team_int, final_opp_int = outcome_aware_postprocess(
        test_pred_team_raw,
        test_pred_opp_raw,
        BEST_PP_PARAMS,
        prior_matrix=full_prior_matrix,
    )
    active_mode = 'outcome_aware'
else:
    final_team_int, final_opp_int = round_team_int, round_opp_int
    active_mode = 'round_clip'

submission = sample[['Id']].copy()
submission['team_goals'] = final_team_int
submission['opp_goals'] = final_opp_int

assert submission.shape == sample.shape
assert submission['Id'].equals(sample['Id'])
submission.to_csv(SUBMISSION_PATH, index=False)

changed_rows = ((submission['team_goals'] != submission_round['team_goals']) | (submission['opp_goals'] != submission_round['opp_goals'])).sum()

print('Saved Jalur A submission:', SUBMISSION_PATH)
print('Saved round+clip backup:', SUBMISSION_ROUNDCLIP_PATH)
print('Active postprocess mode:', active_mode)
print('Best validation blend AW-MAE:', best['score'])
if 'BEST_PP_SCORE' in globals():
    print('Best validation Jalur A AW-MAE:', BEST_PP_SCORE)
if 'BEST_PP_PARAMS' in globals():
    print('Best postprocess params:', BEST_PP_PARAMS)
print('Changed rows vs round+clip:', changed_rows)
print('Submission shape:', submission.shape)

display(submission.head())
display(submission[['team_goals', 'opp_goals']].describe())

print('distribusi skor')
print('team_goals')
print(submission['team_goals'].value_counts())
print('opp_goals')
print(submission['opp_goals'].value_counts())

print('distribusi pasangan skor')
display(submission[['team_goals', 'opp_goals']].value_counts().head(30).to_frame('count'))

Saved Jalur A submission: C:\Users\Gusti Jogish\Downloads\Gammafest\experiments\percobaan 5 - simple baseline\submission_full_ensemble_jalur_a.csv
Saved round+clip backup: C:\Users\Gusti Jogish\Downloads\Gammafest\experiments\percobaan 5 - simple baseline\submission_full_ensemble_roundclip.csv
Active postprocess mode: outcome_aware
Best validation blend AW-MAE: 3.083897760873677
Best validation Jalur A AW-MAE: 3.002770884133499
Best postprocess params: {'max_score': 6, 'draw_margin': 0.0, 'outcome_weight': 0.35, 'gd_weight': 0.03, 'prior_weight': 0.06, 'mode': 'outcome_aware'}
Changed rows vs round+clip: 12073
Submission shape: (42422, 3)


,Id,team_goals,opp_goals
0,M034984_Seychelles,1,2
1,M034984_Mauritius,2,1
2,M034985_Comoros,1,2
3,M034985_Maldives,2,1
4,M034986_Réunion,1,1


,team_goals,opp_goals
count,42422.000000,42422.000000
mean,1.454434,1.459549
std,0.976509,0.990563
min,0.000000,0.000000
25%,1.000000,1.000000
50%,1.000000,1.000000
75%,2.000000,2.000000
max,6.000000,6.000000


distribusi skor
team_goals
team_goals
1    19820
2    13229
0     5003
3     2825
4      981
5      361
6      203
Name: count, dtype: int64
opp_goals
opp_goals
1    19609
2    13162
0     5134
3     2890
4     1042
5      364
6      221
Name: count, dtype: int64
distribusi pasangan skor


count
team_goals opp_goals       
1          2          11867
2          1          11850
1          1           5026
           3           1490
3          0           1387
           1           1355
0          3           1329
2          0           1270
0          2           1176
1          0           1174
0          1           1137
           4            820
4          0            772
5          0            329
0          5            322
1          4            219
0          6            219
4          1            208
6          0            202
3          2             82
2          3             70
1          5             42
2          2             36
5          1             32
2          4              3
1          6              2
3          3              1
4          2              1
6          1              1

## 13. Notes untuk eksperimen berikutnya

Kalau Jalur A belum naik banyak dari 3.062, kemungkinan gain berikutnya ada di:
1. classifier khusus win/draw/loss,
2. threshold calibration per tournament/gender,
3. reconstruct historical features untuk test,
4. hybrid external lookup kalau rules memperbolehkan.